# 02 - Análisis Exploratorio de Datos (EDA)

Proyecto: Radiografía de Steam (EIN092B Visualización, 2026-II). Avance 2.

Este notebook desarrolla el EDA que respalda la aplicación Streamlit. Sigue la secuencia: estructura y calidad, análisis univariado, relaciones entre variables, componente temporal, robustez e información complementaria. Cada visualización se acompaña de su observación e interpretación.

Flujo general: Pregunta, Datos, EDA, Visualización, Hallazgos.

## 0. Pregunta y contexto

Pregunta: ¿cómo se relacionan las características de un juego (género, precio, idiomas, soporte multiplataforma y funcionalidades comunitarias) con su recepción y compromiso, entre 2014 y 2024?

- X: características de publicación y diseño.
- Y: proporción de reseñas positivas, tiempo de juego, volumen de reseñas.
- T: año de lanzamiento, acotado a 2014 a 2024.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import config, data_loading, viz

crudo = data_loading.cargar_crudo(config.RUTA_DATOS_RAW)
df = data_loading.limpiar(crudo)

# No se restringe la muestra: se usan todos los juegos de la ventana.
# Los analisis de recepcion toman automaticamente los juegos con resenas.
con_y = df.dropna(subset=['pct_positivas'])

print('Archivo original:', len(crudo))
print('Ventana 2014 a 2024:', len(df))
print('Con al menos una resena:', len(con_y))

## 1. Estructura y calidad de los datos

In [ ]:
data_loading.resumen_calidad(crudo).head(10)

In [ ]:
# Problemas de calidad cuantificados
print('Juegos sin resenas:', int((df['total_resenas'] == 0).sum()))
print('Recepcion exactamente 100 %:', round((con_y['pct_positivas'] == 1).mean(), 3))
print('Idiomas marcados en bloque (> 29):', int(df['idiomas_atipico'].sum()))
print('Juegos con tiempo de juego registrado:', round((df['playtime_mediano'] > 0).mean(), 3))
print('Juegos de mas de 100 USD:', int((df['precio'] > 100).sum()))

**Observación.** El archivo tiene 125.855 juegos; 95.595 caen en la ventana 2014 a 2024 y 20.425 de ellos (21 %) no tienen ninguna reseña. Las columnas con más faltantes son Score rank (100 %), Tags (34 %) y Categories (7 %), ninguna crítica para la pregunta.

**Problemas encontrados y decisiones.**

- El encabezado original fusiona las columnas Discount y DLC count, lo que desplaza todos los valores una posición. La carga separa ese nombre.
- No se excluye ningún juego por precio ni por número de reseñas. La proporción de positivas solo existe con reseñas (75.170 juegos), y con muy pocas reseñas toma valores extremos: el 17,6 % tiene exactamente 100 %. Por eso se agrega una prueba de robustez con un mínimo de reseñas.
- 1.804 juegos declaran más de 29 idiomas (hasta 103, todos los que permite la tienda). Se conservan y se muestran como grupo aparte.
- Solo el 23 % de los juegos tiene tiempo de juego registrado; se analiza como información complementaria sobre ese subgrupo.
- 1.075 registros declaran solo géneros de software, aunque la fuente dice incluir únicamente juegos. Se conservan y se declaran como limitación.
- Los géneros vienen casi siempre en orden alfabético (83 % de las listas), así que el análisis por género es multietiqueta.

## 2. Análisis univariado

In [ ]:
viz.distribucion_y(df).show()

**Observación.** La mediana de reseñas positivas es 81 % y el 53 % de los juegos con reseñas supera el 80 %. Se ven picos en 0 % y 100 %, propios de los juegos con muy pocas reseñas.

**Interpretación.** La mayoría de los juegos es bien valorada; el interés analítico está en qué características acompañan a la cola baja.

In [ ]:
viz.distribucion_precio(df, log_y=True).show()

**Observación.** El precio mediano es 1,99 USD, el 21 % es gratuito y el 75 % cuesta 4,99 USD o menos. Solo 263 juegos superan los 100 USD (máximo 999,98).

**Interpretación.** El mercado se concentra en segmentos económicos, lo que condiciona cualquier comparación por precio.

In [ ]:
viz.frecuencia_generos(df).show()

**Observación.** Indie es el género más declarado (64.473 juegos), seguido de Casual (39.641), Action (37.301) y Adventure (36.237).

**Interpretación.** Un juego puede declarar varios géneros, por lo que las barras no suman el total. Casual aparece segundo porque se incluyen los juegos sin reseñas, muchos de ellos casuales.

In [ ]:
data_loading.estadisticas_descriptivas(df)

**Observación.** El total de reseñas es muy asimétrico: la media (1.173) es cien veces la mediana (12), porque unos pocos juegos concentran la mayor parte de las reseñas. Por eso el análisis usa medianas y correlación de Spearman.

## 3. Relaciones entre variables

In [ ]:
viz.recepcion_por_genero(df, top=12).show()

**Observación.** Entre los 12 géneros con más juegos, Casual tiene la mediana más alta (83 %) y Massively Multiplayer la más baja (66 %).

**Interpretación.** El género se asocia a la recepción, pero las cajas se traslapan mucho: por sí solo no la determina.

In [ ]:
viz.precio_vs_recepcion(df, con_tendencia=True).show()
viz.recepcion_por_tramo_precio(df).show()

**Observación.** La correlación entre precio y recepción es débil (Spearman 0,06). Por tramos, la mediana sube de 79 % (gratis) a 84 % (2 a 5 USD), se mantiene cerca de 80 % hasta 30 USD y cae a 69 % entre 40 y 60 USD y a 44 % sobre 100 USD (150 juegos).

**Interpretación.** La relación no es lineal: forma una U invertida. Los juegos de más de 100 USD mezclan software profesional bien valorado con juegos de bajo esfuerzo a 199,99 USD que reciben malas reseñas.

In [ ]:
viz.recepcion_por_comunidad(df).show()

**Observación.** Los juegos con multijugador o Steam Workshop tienen una mediana de 80 %, frente a 82 % de los que no tienen.

**Interpretación.** Contra lo esperado, la comunidad no se asocia a mejor recepción. Una hipótesis es que el multijugador expone el juego a más críticas (servidores, emparejamiento, comunidad activa).

In [ ]:
viz.idiomas_vs_popularidad(df).show()

**Observación.** La mediana de reseñas pasa de 9 con un idioma a 141 con 13 a 20 idiomas. Los grupos marcados en bloque caen a 16 (29 idiomas) y 0 (más de 29).

**Interpretación.** Los idiomas no se relacionan con la recepción (Spearman 0,00), pero sí con la popularidad. Los grupos en bloque no reflejan una localización real.

In [ ]:
viz.matriz_correlacion(df).show()

**Observación.** Ninguna variable tiene una correlación fuerte con la recepción; la mayor es el número de sistemas operativos (0,08). En cambio, idiomas y total de reseñas se correlacionan en 0,25.

**Interpretación.** Las características de publicación se asocian más a la popularidad que a la valoración.

## 4. Componente temporal (T)

In [ ]:
viz.evolucion_recepcion(df).show()
df.groupby('anio')['total_resenas'].median()

**Observación.** La recepción mediana sube de 74 % en 2014 a 89 % en 2024, mientras la mediana de reseñas baja de 314 a 3.

**Interpretación.** La tendencia se lee con cautela: los juegos recientes tienen muy pocas reseñas, las primeras suelen venir de jugadores entusiastas, y los juegos antiguos que siguen visibles acumulan también las críticas de largo plazo.

## 5. Prueba de robustez

Se repiten los resultados principales exigiendo un mínimo de 10 reseñas.

In [ ]:
r = df[df['total_resenas'] >= config.MIN_RESENAS_ROBUSTEZ].dropna(subset=['pct_positivas'])
pd.DataFrame({
    'Muestra completa': [len(con_y), con_y['pct_positivas'].median(),
                         con_y['precio'].corr(con_y['pct_positivas'], method='spearman'),
                         con_y.groupby('comunidad')['pct_positivas'].median().diff().iloc[-1]],
    'Con 10 o mas resenas': [len(r), r['pct_positivas'].median(),
                             r['precio'].corr(r['pct_positivas'], method='spearman'),
                             r.groupby('comunidad')['pct_positivas'].median().diff().iloc[-1]],
}, index=['Juegos con resenas', 'Mediana de recepcion', 'Spearman precio', 'Diferencia comunidad']).round(3)

**Observación.** Con 10 o más reseñas quedan 51.616 juegos; la mediana se mantiene en 81 %, la correlación con el precio sube a 0,11 y la diferencia por comunidad pasa a -2,6 puntos.

**Interpretación.** Los hallazgos son robustos: cambian las magnitudes, pero no la dirección.

## 6. Información complementaria

In [ ]:
viz.tiempo_por_tramo_precio(df).show()
viz.tiempo_por_genero(df).show()

**Observación.** Solo el 23 % de los juegos tiene tiempo de juego registrado, con una mediana de 3,4 horas. Esos juegos tienen una mediana de 292 reseñas, frente a 6 del resto. Las horas suben de 1,5 (gratis) a 21,3 (40 a 60 USD); RPG y Strategy son los géneros más largos (4,4 y 4,3 h) y Free To Play el más corto (1,4 h).

**Interpretación.** El registro se concentra en juegos con más audiencia. Dentro de ese grupo, el precio se asocia a más horas (Spearman 0,28), pero las horas casi no se relacionan con la recepción (0,13).

In [ ]:
viz.idiomas_mas_frecuentes(df, destacar='Spanish - Latin America').show()
viz.comparar_idioma(df, 'Spanish - Latin America')

**Observación.** El inglés está en el 96 % de los juegos y el chino simplificado es el segundo idioma (26 %). El español latinoamericano llega solo al 6 %, frente al 21 % del español de España. Los juegos que lo soportan tienen 40 reseñas medianas frente a 14, y 85 % de recepción frente a 81 %.

**Interpretación.** La comparación es descriptiva: localizar a un idioma suele ser propio de juegos con más presupuesto.

In [ ]:
viz.recepcion_por_n_generos(df).show()

**Observación.** Los juegos con 1 o 2 géneros tienen una recepción mediana de 83 %; los que declaran más de 8 bajan a 66 %.

**Interpretación.** Marcar casi todos los géneros parece una forma de ganar visibilidad, y se asocia a peor valoración.

## 7. Hallazgos preliminares

1. La recepción está concentrada en valores altos: la mediana es 81 % y el 53 % de los juegos con reseñas supera el 80 %.
2. Ninguna característica explica por sí sola la recepción: las correlaciones con precio, idiomas y sistemas operativos son menores a 0,15.
3. El precio tiene una relación no lineal (U invertida): los juegos de 2 a 15 USD son los mejor valorados y los de más de 40 USD caen con claridad.
4. La comunidad no mejora la recepción (80 % frente a 82 %).
5. Los idiomas se asocian a la popularidad, no a la valoración.
6. La tendencia temporal está condicionada por el volumen de reseñas.
7. Los resultados son robustos al exigir un mínimo de reseñas.
8. Hay señales de metadatos inflados (demasiados géneros o idiomas) asociadas a juegos de bajo esfuerzo.

**Cómo refinan la pregunta.** Las características de publicación parecen explicar mejor cuánta atención recibe un juego (popularidad) que qué tan bien es valorado. Para el Avance 3 se propone separar ambos objetivos y controlar por el año de lanzamiento y el volumen de reseñas.